# Student Performance Prediction using Linear Regression

**What this does:** Predicts a student's final exam score (G3) based on inputs like study time, past failures, absences, parental education, etc.

**Approach:** Linear Regression — finds the relationship between student attributes and final grade, then predicts grade for new inputs.

**Dataset:** UCI Student Performance Dataset (Math subject) — real-world data, 395 students, 30+ features.


## Step 1: Download Dataset

In [ ]:
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
zip_path = "student.zip"

if not os.path.exists("student-mat.csv"):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(".")
    print("Downloaded and extracted.")
else:
    print("Already present.")


## Step 2: Load and Inspect Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("student-mat.csv", sep=";")
print("Shape:", df.shape)
df.head()


In [ ]:
# Target variable: G3 = final grade (0-20 scale)
print("Grade distribution (G3):")
print(df['G3'].describe())


## Step 3: Feature Selection

We select features that logically and statistically influence final grade.
- **Numeric features:** study time, past failures, absences, G1 (period 1 grade), G2 (period 2 grade)
- **Categorical features:** school support, paid classes, internet access — encoded as 0/1

**Honest note:** G1 and G2 (period grades) are the strongest predictors of G3. Including them is valid — in a real school system, you'd use mid-term performance to predict finals. If your brief says "predict without using prior grades," drop G1 and G2 and model accuracy will drop significantly. Know that tradeoff.


In [ ]:
# Select features
features = ['studytime', 'failures', 'absences', 'G1', 'G2',
            'schoolsup', 'paid', 'internet', 'Medu', 'Fedu']

target = 'G3'

# Encode binary categorical columns (yes/no -> 1/0)
binary_cols = ['schoolsup', 'paid', 'internet']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

X = df[features]
y = df[target]

print("Features shape:", X.shape)
print("\nFeature columns:")
print(X.columns.tolist())


## Step 4: Exploratory Data Analysis

In [ ]:
# Correlation heatmap - shows which features matter most
plt.figure(figsize=(10, 6))
corr_data = df[features + [target]].corr()
sns.heatmap(corr_data, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Matrix - Student Features vs Final Grade (G3)")
plt.tight_layout()
plt.show()


In [ ]:
# Distribution of final grades
plt.figure(figsize=(8, 4))
plt.hist(df['G3'], bins=20, color='steelblue', edgecolor='black')
plt.title("Distribution of Final Grades (G3)")
plt.xlabel("Grade (0-20)")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.show()


In [ ]:
# G2 vs G3 — period 2 grade is strongest single predictor
plt.figure(figsize=(7, 5))
plt.scatter(df['G2'], df['G3'], alpha=0.5, color='steelblue')
plt.xlabel("Period 2 Grade (G2)")
plt.ylabel("Final Grade (G3)")
plt.title("G2 vs G3 — Mid-term vs Final Grade")
plt.tight_layout()
plt.show()


## Step 5: Train-Test Split and Train Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")

# Scale features (helps linear regression converge cleanly)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train
model = LinearRegression()
model.fit(X_train_scaled, y_train)
print("\nModel trained.")


## Step 6: Evaluate Model

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test_scaled)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=" * 35)
print("       Model Evaluation Results")
print("=" * 35)
print(f"  R² Score  : {r2:.4f}  (higher = better, max 1.0)")
print(f"  MAE       : {mae:.4f}  (avg error in grade points)")
print(f"  RMSE      : {rmse:.4f}  (penalizes large errors more)")
print("=" * 35)


In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='black', linewidths=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel("Actual Grade (G3)")
plt.ylabel("Predicted Grade")
plt.title("Actual vs Predicted Final Grade")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Feature importance (coefficients after scaling = fair comparison)
coef_df = pd.DataFrame({
    'Feature': features,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(8, 5))
colors = ['green' if c > 0 else 'red' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.xlabel("Coefficient (impact on final grade)")
plt.title("Feature Importance — Linear Regression Coefficients")
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))


## Step 7: Predict for a New Student

Enter a student's details and get a predicted final grade.

In [ ]:
def predict_student(studytime, failures, absences, G1, G2,
                    schoolsup, paid, internet, Medu, Fedu):
    """
    studytime  : 1=<2hrs, 2=2-5hrs, 3=5-10hrs, 4=>10hrs per week
    failures   : number of past class failures (0-3)
    absences   : number of school absences
    G1, G2     : grades from period 1 and 2 (0-20)
    schoolsup  : extra school support? 1=yes, 0=no
    paid       : paid extra classes? 1=yes, 0=no
    internet   : internet at home? 1=yes, 0=no
    Medu, Fedu : mother/father education level (0=none to 4=higher ed)
    """
    input_data = pd.DataFrame([[studytime, failures, absences, G1, G2,
                                schoolsup, paid, internet, Medu, Fedu]],
                               columns=features)
    input_scaled = scaler.transform(input_data)
    predicted = model.predict(input_scaled)[0]
    predicted = max(0, min(20, round(predicted, 1)))  # clip to valid grade range

    print(f"Predicted Final Grade (G3): {predicted} / 20")
    if predicted >= 16:
        print("Performance Band: Excellent")
    elif predicted >= 12:
        print("Performance Band: Good")
    elif predicted >= 10:
        print("Performance Band: Passing")
    else:
        print("Performance Band: At Risk of Failure")
    return predicted


# Example student
predict_student(
    studytime=2,
    failures=0,
    absences=4,
    G1=12,
    G2=13,
    schoolsup=0,
    paid=1,
    internet=1,
    Medu=3,
    Fedu=2
)


In [ ]:
# Try a struggling student
predict_student(
    studytime=1,
    failures=2,
    absences=20,
    G1=7,
    G2=6,
    schoolsup=0,
    paid=0,
    internet=0,
    Medu=1,
    Fedu=1
)


## Viva / Report Notes

**Why Linear Regression?**
The target (G3) is a continuous number (0–20), not a category. Linear regression is the correct baseline model for continuous output prediction. It's interpretable — you can directly read which features increase or decrease predicted grade from the coefficients.

**Why not Logistic Regression / Decision Tree?**
Logistic Regression is for classification (pass/fail), not score prediction. A Decision Tree would also work here and handles non-linearity better — if an examiner asks "how would you improve this," Decision Tree or Random Forest is a legitimate and honest upgrade.

**Biggest limitation to be upfront about:**
G1 and G2 (mid-term grades) dominate predictions. Without them, model accuracy drops significantly. If the use case is "predict early before any grades exist," you'd drop G1/G2 and the model becomes weaker — that's a real limitation, not a bug to hide.

**What R² means in plain language:**
An R² of 0.85 means your model explains 85% of the variation in final grades. The remaining 15% is driven by things not in your data (mood, health, exam difficulty, etc.).

**Don't claim this is "AI" or "deep learning."**
This is classical statistics — linear regression. It's a valid, widely-used ML technique. Say "machine learning using Linear Regression" — that's accurate. Saying "AI-powered student prediction" for a linear regression is overclaiming and a viva panel will push back on it.
